# Colab Embed Chunks for Legal RAG (Vietnamese Labor Law)

Notebook này dùng GPU T4 miễn phí trên Google Colab để tính toán dense embeddings cho toàn bộ kho văn bản luật (`chunks.parquet`) và xuất file `embeddings.parquet` về máy tính cục bộ.

### 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

### 2. Cài đặt các thư viện cần thiết

In [ ]:
!pip install sentence-transformers pandas pyarrow pydantic pydantic-settings pyyaml

### 3. Tải lên file `chunks.parquet`
Bạn hãy tải file `data/processed/chunks.parquet` từ máy tính lên thư mục hiện tại của Colab.

In [ ]:
from google.colab import files
import os

if not os.path.exists('chunks.parquet'):
    print('Hãy chọn tải lên file chunks.parquet:')
    uploaded = files.upload()
else:
    print('File chunks.parquet đã có sẵn.')

### 4. Chạy Embedding trên GPU

In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
import time

model_name = 'AITeamVN/Vietnamese_Embedding'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading model {model_name} on {device}...')
model = SentenceTransformer(model_name, device=device)
model.max_seq_length = 2048

chunks_df = pd.read_parquet('chunks.parquet')
print(f'Loaded {len(chunks_df)} chunks.')

texts = chunks_df['embed_text'].astype(str).tolist()
cids = chunks_df['chunk_id'].astype(str).tolist()

start = time.time()
vectors = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
elapsed = time.time() - start
print(f'Embedding finished in {elapsed:.1f}s ({len(texts)/elapsed:.1f} chunks/s)')

out_df = pd.DataFrame({
    'chunk_id': cids,
    'vector': [v.tolist() for v in vectors]
})
out_df.to_parquet('embeddings.parquet', index=False)
print(f'Saved embeddings.parquet ({len(out_df)} rows).')

### 5. Tải file `embeddings.parquet` về máy

In [ ]:
from google.colab import files
files.download('embeddings.parquet')